In [ ]:
!pip install google-generativeai pandas

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 34.6 MB/s  0:00:00
   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   ---------------------------------------- 4.9/4.9 MB 66.7 MB/s  0:00:00
   ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
   ---------------------------------------  15.2/15.3 MB 91.3 MB/s eta 0:00:01
   ---------------------------------------- 15.3/15.3 MB 73.4 MB/s  0:00:00

  Attempting uninstall: requests

    Found existing installation: requests 2.32.5

    Uninstalling requests-2.32.5:

      Successfully uninstalled requests-2.32.5

   -- ---------

In [11]:
!pip install google-generativeai

In [3]:
import pandas as pd


thread_df = pd.read_csv("../../02_Data/processed/comments_merged_thread_add_tag.csv")


In [4]:
thread_df

,Unnamed: 0,projectID,thread_id,group,comment_ids,n_comments,merged_comment,creator_id,is_pledge_master,is_backer,is_prior_backer,is_pathfinder,likes
0,0,5787,1691589,0,[1691589],1,WELCOME TO EDEN! Please check the FAQ and the...,1,0,0,0,0,26
1,1,5787,1725920,0,"[1725920, 1726005]",2,@BlackSiteStudios Could a person use larger di...,1,1,1,0,0,0
2,2,5787,1725876,0,[1725876],1,about an hour left to go,0,1,1,1,0,1
3,3,5787,1725671,0,[1725671],1,Man 2 hours to go worth staying up to midnight...,0,0,0,0,0,3
4,4,5787,1725512,0,[1725512],1,5 hours to go! Hold on to your butts!,0,0,1,1,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
73434,73434,1143,133497,1,"[133497, 133510, 134222]",3,"Hello. Will it be in Spanish? and if not, woul...",1,2,0,0,0,4
73435,73435,1143,133314,1,"[133314, 133507]",2,Not sure if people get notifications for repli...,1,1,1,0,0,4
73436,73436,1143,133207,1,"[133207, 133219]",2,The project looks interesting! Who is the desi...,1,0,0,0,0,0
73437,73437,1143,133159,1,"[133159, 133202, 133500]",3,"Hi, is there the future possibility of a solo ...",1,1,0,0,0,11


In [23]:
thread_df.loc[thread_df['projectID']==1476]

,Unnamed: 0,projectID,thread_id,group,comment_ids,n_comments,merged_comment,creator_id,is_pledge_master,is_backer,is_prior_backer,is_pathfinder,likes


In [6]:
thread_df["merged_comment"][1]

"@BlackSiteStudios Could a person use larger dinosaur figures as long as they scale up the battle mat area to fit them?|Sure! The game is designed for a 2x2ft play area, so you'll need to do some balance yourself. The rules will tell you what base size a particular dinosaur will have, so if the models you have fit on that, then you should be good."

In [1]:
# 테스트 30개
#test_df = thread_df.sample(n=30, random_state=42).copy() 

In [ ]:
import time
import pandas as pd
import google.generativeai as genai
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# API 설정 (api키는 각자 발급받아 .txt 파일로 저장 후 진행해야함)
try:
    with open("api_key.txt", "r") as file:
        api_key = file.read().strip() 
    genai.configure(api_key=api_key)
except FileNotFoundError:
    print("api_key.txt 파일 오류.")
    exit()

# 비용 가장 저렴
model = genai.GenerativeModel('gemini-3.1-flash-lite')

target_df = thread_df.copy().reset_index(drop=True)

# 프롬포트 입력
system_prompt = """You are an expert community manager for a global crowdfunding platform.
Your task is to analyze user comments and classify the core intent into exactly ONE of the following categories:

[Categories]
1. Shipping_Fulfillment
2. Product_Question
3. Praise_Support
4. Complaint_Refund
5. Suggestion_Idea
6. Spam_Irrelevant

[Rules]
- Read the entire conversation thread and select the ONE category that best fits the CORE intent.
- Lines starting with '|' are replies to the main comment.
- DO NOT provide any explanation.
- ONLY output the exact name of the category."""

# 시간 오래 걸려 병렬처리
def classify_row(args):
    seq_idx, row = args
    try:
        full_prompt = f"{system_prompt}\n\n[Thread Text]\n{row['merged_comment']}"
        response = model.generate_content(full_prompt)
        return seq_idx, response.text.strip()
    except Exception:
        return seq_idx, "Error"

print(f" 총 {len(target_df)}개 분류 시작")

results_dict = {}

# 원하는 최종 컬럼 순서 지정
#['Unnamed: 0', 'projectID', 'thread_id', 'group', 'comment_ids',
#       'n_comments', 'merged_comment', 'creator_id', 'is_pledge_master',
#       'is_backer', 'is_prior_backer', 'is_pathfinder', 'likes']
selected_columns = ['thread_id', 'projectID', 'group', 'n_comments', 'merged_comment', 'creator_id', 'is_pledge_master','is_backer', 'is_prior_backer', 'is_pathfinder', 'likes','category']

# 50개의 스레드를 동시에 사용하여 구글 서버에 요청
with ThreadPoolExecutor(max_workers=50) as executor:
    futures = [executor.submit(classify_row, (seq_idx, row)) for seq_idx, (_, row) in enumerate(target_df.iterrows())]
    
    for future in tqdm(futures, total=len(target_df), desc="분류 진행중"):
        seq_idx, category = future.result()
        results_dict[seq_idx] = category

# 최종 저장
target_df['category'] = [results_dict[i] for i in range(len(target_df))]
final_df = target_df[selected_columns]

final_df.to_csv("../../02_Data/processed/final_all_results.csv", index=False, encoding="utf-8-sig")
print("----완료-----")

In [14]:
target_df.groupby('group')['category'].value_counts()
target_df.groupby(['group', 'category']).size()
result_df = target_df.groupby('group')['category'].value_counts().reset_index(name='count')
result_df

,group,category,count
0,0,Product_Question,13816
1,0,Suggestion_Idea,9672
2,0,Praise_Support,6431
3,0,Shipping_Fulfillment,2184
4,0,Spam_Irrelevant,1374
5,0,Complaint_Refund,1085
6,0,Please provide the thread text you would like ...,5
7,0,Please provide the [Thread Text] you would lik...,1
8,0,Please provide the full thread text you would ...,1
9,0,Please provide the text you would like me to a...,1


In [20]:
target_df.head()

,Unnamed: 0,projectID,thread_id,group,comment_ids,n_comments,merged_comment,creator_id,is_pledge_master,is_backer,is_prior_backer,is_pathfinder,likes,category
0,0,5787,1691589,0,[1691589],1,WELCOME TO EDEN! Please check the FAQ and the...,1,0,0,0,0,26,Praise_Support
1,1,5787,1725920,0,"[1725920, 1726005]",2,@BlackSiteStudios Could a person use larger di...,1,1,1,0,0,0,Product_Question
2,2,5787,1725876,0,[1725876],1,about an hour left to go,0,1,1,1,0,1,Spam_Irrelevant
3,3,5787,1725671,0,[1725671],1,Man 2 hours to go worth staying up to midnight...,0,0,0,0,0,3,Praise_Support
4,4,5787,1725512,0,[1725512],1,5 hours to go! Hold on to your butts!,0,0,1,1,0,5,Praise_Support


In [ ]:
final_df['category'].value_counts()

category
Product_Question                                                     32538
Suggestion_Idea                                                      19122
Praise_Support                                                       10908
Shipping_Fulfillment                                                  5626
Complaint_Refund                                                      2984
Spam_Irrelevant                                                       2248
Please provide the thread text you would like me to analyze.             7
Please provide the [Thread Text] you would like me to analyze.           1
Please provide the thread text, and I will classify it for you.          1
Please provide the text you would like me to classify.                   1
Please provide the text you would like me to analyze.                    1
Please provide the full thread text you would like me to analyze.        1
Please provide the text you would like me to analyze!                    1
Name: count, dty